# 03 · Ingest and build Layers A, B and C

**Runs on: the store box** (the one with `kubectl`). No GPU needed — everything
left is aggregation, clustering and LLM calls.

Downloads the annotated parquets, upserts them into the platform's chunk index,
then builds the graph into the platform's Neo4j. Layer B and Layer C call Groq;
nothing here runs a model locally.

## 0. Preflight

```bash
kubectl --context k8s-w -n wf-prod port-forward svc/elastic 9200:9200 &
kubectl --context k8s-w -n wf-prod port-forward svc/neo4j   7687:7687 &

export ELASTICSEARCH_URL=http://localhost:9200
export NEO4J_URL=bolt://localhost:7687
export NEO4J_PASSWORD=...           # the password half of NEO4J_AUTH
export GROQ_API_KEY=...
export S3_ACCESS_KEY=root
export S3_SECRET_KEY=...
export FS_RUN_ID=2026-09-21-full    # the same run as the other two notebooks
```

**Two things this notebook states rather than solves.**

`GraphStoreConfig` has no `database` field and the Neo4j store opens
`session()` with no argument, so this writes into the **default** database —
the one holding RecipeWrangler's 88k ingredients and 7.6k recipes. The labels
are disjoint (`:Shelf`/`:Theme`/`:Card` versus `:Recipe`/`:Ingredient`) so they
coexist, but this is not isolated. Point `NEO4J_URL` at a separate instance if
you want that today.

Layer B labels come from one LLM call each, so theme ids do not reproduce
across runs. **Any rebuild orphans existing cards**, which is why Layer C
always follows Layer B here rather than being skippable.

In [ ]:
import os, sys, time, json
from pathlib import Path

REQUIRED = ["ELASTICSEARCH_URL", "NEO4J_URL", "NEO4J_PASSWORD",
            "GROQ_API_KEY", "S3_ACCESS_KEY", "S3_SECRET_KEY", "FS_RUN_ID"]
missing = [v for v in REQUIRED if not os.environ.get(v)]
if missing:
    raise SystemExit(f"missing environment: {', '.join(missing)}")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
WORK = DATA / "handoff" / os.environ["FS_RUN_ID"]
(WORK / "annotated").mkdir(parents=True, exist_ok=True)

ES_URL = os.environ["ELASTICSEARCH_URL"]
NEO4J_URL = os.environ["NEO4J_URL"]
CHUNK_INDEX = os.environ.get("FS_CHUNK_INDEX", "foodscholar_chunks")
CARD_INDEX  = os.environ.get("FS_CARD_INDEX", "foodscholar_cards")
print(WORK)

In [ ]:
import os, json, hashlib
from pathlib import Path

try:
    import boto3
    from botocore.client import Config as BotoConfig
except ImportError as e:
    raise SystemExit("pip install boto3  # S3 handoff between the GPU box and the store box") from e

S3_ENDPOINT = os.environ.get("S3_ENDPOINT", "https://s3.wisefood-project.eu")
S3_BUCKET   = os.environ.get("S3_BUCKET", "foodscholar-graph-build")
RUN_ID      = os.environ["FS_RUN_ID"]          # same string on both machines

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    # MinIO speaks path-style; virtual-host style would resolve
    # <bucket>.s3.wisefood-project.eu, which has no DNS record.
    config=BotoConfig(signature_version="s3v4", s3={"addressing_style": "path"}),
)

def s3_key(*parts: str) -> str:
    return "/".join(["runs", RUN_ID, *parts])

def s3_exists(key: str) -> bool:
    try:
        s3.head_object(Bucket=S3_BUCKET, Key=key)
        return True
    except Exception:
        return False

def s3_put(local: Path, key: str) -> None:
    s3.upload_file(str(local), S3_BUCKET, key)

def s3_get(key: str, local: Path) -> Path:
    local.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(S3_BUCKET, key, str(local))
    return local

def s3_list(prefix: str) -> list[str]:
    keys, token = [], None
    while True:
        kw = {"Bucket": S3_BUCKET, "Prefix": prefix}
        if token:
            kw["ContinuationToken"] = token
        resp = s3.list_objects_v2(**kw)
        keys.extend(o["Key"] for o in resp.get("Contents", []))
        if not resp.get("IsTruncated"):
            return sorted(keys)
        token = resp["NextContinuationToken"]

print(f"s3  {S3_ENDPOINT}/{S3_BUCKET}")
print(f"run {RUN_ID}")

## 1. Pull the annotated shards

In [ ]:
manifest = json.loads(s3_get(s3_key("manifest.json"), WORK / "manifest.json").read_text())
expected = len(manifest["shards"])
keys = s3_list(s3_key("annotated"))
print(f"{len(keys)} of {expected} annotated shards on s3")
if len(keys) < expected:
    raise SystemExit("incomplete — finish 02_annotate on the GPU box first")

for key in keys:
    local = WORK / "annotated" / Path(key).name
    if not local.exists():
        s3_get(key, local)
parquets = sorted((WORK / "annotated").glob("shard_*.parquet"))
print(f"{len(parquets)} parquets local")

## 2. Configure — the platform's stores

`annotate` config is still present because `config_hash` covers it and the
hash is stamped on every artifact. Nothing here runs GLiNER.

In [ ]:
from foodscholar import FoodScholar

CONFIG = {
    "corpus": {"chunks_path": str(WORK / "chunks.parquet"), "annotated_snapshot_path": None},
    "ontology": {"foodon_path": str(DATA / "foodon.owl"),
                 "cache_path": str(DATA / "foodon_cache.parquet"),
                 "include_imports": False},
    "storage": {
        "chunk_store": {"backend": "elastic", "url": ES_URL, "index": CHUNK_INDEX},
        "card_store":  {"backend": "elastic", "url": ES_URL, "index": CARD_INDEX},
        "graph_store": {"backend": "neo4j", "url": NEO4J_URL,
                        "user": os.environ.get("NEO4J_USER", "neo4j"),
                        "password": os.environ["NEO4J_PASSWORD"]},
    },
    "llm": {"provider": "groq", "model": os.environ.get("FS_LLM_MODEL", "openai/gpt-oss-120b")},
}

fs = FoodScholar.from_config(CONFIG)
print(json.dumps(fs.info(), indent=2, default=str))
print("config hash:", fs.config_hash)

## 3. Ingest the annotated chunks

`load_chunks` on a `.parquet` reads mentions, entity links **and** embeddings
straight back — nothing is recomputed here. Idempotent by chunk id, so a
re-run overwrites rather than duplicates.

In [ ]:
from elasticsearch import Elasticsearch
es = Elasticsearch(hosts=ES_URL)

def count(index, body=None):
    return int(es.count(index=index, body=body)["count"]) if es.indices.exists(index=index) else 0

before = count(CHUNK_INDEX)
t0 = time.perf_counter()
for i, p in enumerate(parquets, 1):
    fs.load_chunks(p)
    if i % 20 == 0 or i == len(parquets):
        print(f"  {i}/{len(parquets)} shards ingested", flush=True)
es.indices.refresh(index=CHUNK_INDEX)
after = count(CHUNK_INDEX)
print(f"\nchunk index: {before} -> {after} (+{after-before}) in {(time.perf_counter()-t0)/60:.1f} min")

In [ ]:
# The annotation actually landed, and carried its links across the handoff.
annotated = count(CHUNK_INDEX, {"query": {"term": {"enrichment_version": "annotate-v2"}}})
linked    = count(CHUNK_INDEX, {"query": {"exists": {"field": "foodon_ids"}}})
by_source = {b["key"]: b["doc_count"] for b in es.search(
    index=CHUNK_INDEX, size=0,
    body={"aggs": {"s": {"terms": {"field": "source_type", "size": 10}}}}
)["aggregations"]["s"]["buckets"]}

print(f"annotated : {annotated}")
print(f"linked    : {linked}")
for k, v in sorted(by_source.items()):
    print(f"  {k:10s} {v:7d}")
assert linked > 0, "nothing is linked — the NEL index on the GPU box was broken"

## 4. Embedding backstop

Insurance, not a phase. The annotate runner embedded everything it touched, so
this should find nothing to do — but `only_missing=True` makes asserting that
free, and Layer B silently produces nonsense from unembedded chunks.

In [ ]:
meta = fs.embed(only_missing=True)
print(meta)

## 5. Entities

In [ ]:
meta = fs.build_entities()
print(meta)
print("entities:", len(fs.entities))

## 6. Layer A — the shelf backbone

In [ ]:
meta = fs.build_layer_a()
shelves = fs.graph_store.list_shelves()
by_facet = {}
for s in shelves:
    by_facet[s.facet] = by_facet.get(s.facet, 0) + 1
print(meta)
print(f"shelves: {len(shelves)}")
for f, n in sorted(by_facet.items()):
    print(f"  {f:20s} {n}")

## 7. Attach chunks to shelves

In [ ]:
meta = fs.attach()
print(meta)
print("chunks with a shelf:", count(CHUNK_INDEX, {"query": {"exists": {"field": "shelf_ids"}}}))

## 8. Layer B — themes

**Stop and look before going on.** Layer B is where a bad corpus decision shows
up, and it is the last point at which stopping is free: Layer C spends an LLM
call per theme.

In [ ]:
FACETS = ["foods", "health", "sustainability", "dietary_patterns", "allergies", "nutrients"]

for facet in FACETS:
    t0 = time.perf_counter()
    try:
        meta = fs.build_layer_b(facet=facet)
        print(f"{facet:20s} {meta.record_count:5d} themes  ({time.perf_counter()-t0:.0f}s)", flush=True)
    except Exception as exc:
        print(f"{facet:20s} FAILED: {exc}", flush=True)

themes = fs.graph_store.list_themes()
print(f"\nthemes total: {len(themes)}")
for t in themes[:15]:
    print(f"  {t.theme_id}")

## 9. Layer C — cards

One LLM call per theme, facet by facet so a failure costs one facet.

In [ ]:
for facet in FACETS:
    t0 = time.perf_counter()
    try:
        meta = fs.build_layer_c(facet=facet)
        print(f"{facet:20s} {meta.record_count:5d} cards  ({time.perf_counter()-t0:.0f}s)", flush=True)
    except Exception as exc:
        print(f"{facet:20s} FAILED: {exc}", flush=True)

es.indices.refresh(index=CARD_INDEX)
print("\ncards in the index:", count(CARD_INDEX))

## 10. Verify, then publish

The graph is in Neo4j and Elasticsearch now. The FoodScholar API still serves
the **previous** projection until the browse index is rebuilt — which is what
you want, since a stale graph beats no graph while this runs.

Publish as an admin:

```
POST /api/v1/foodscholar/graph/reindex
```

It reads the whole graph and repoints the alias only on success, so a failed
projection leaves the last good one in place.

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    NEO4J_URL, auth=(os.environ.get("NEO4J_USER", "neo4j"), os.environ["NEO4J_PASSWORD"]))
with driver.session() as session:
    for label in ("Shelf", "Theme", "Card", "Entity"):
        n = session.run(f"MATCH (n:{label}) RETURN count(n) AS c").single()["c"]
        print(f"  :{label:8s} {n}")
    # RecipeWrangler shares this database. If these went to zero, something
    # overwrote rather than added, and that is a restore-from-backup problem.
    for label in ("Recipe", "Ingredient"):
        n = session.run(f"MATCH (n:{label}) RETURN count(n) AS c").single()["c"]
        print(f"  :{label:8s} {n}   (RecipeWrangler — must be unchanged)")
driver.close()

print()
print(f"  chunks        {count(CHUNK_INDEX)}")
print(f"  cards         {count(CARD_INDEX)}")
print(f"  graph_version {fs.config_hash}")
print("\nNow POST /api/v1/foodscholar/graph/reindex as an admin to publish.")